# E08 — ver mundos basta?

A pergunta da volta 4, a A4 da lista do contrato: **ver mundos basta, ou é preciso mexer no mundo
de propósito?** A tentativa natural é acumular mercados --- mais dados, mais identificação. Este
caderno mede se isso acontece.

**O que se mede.** A mesma família de explicações do capítulo anterior, exigindo as quatro
estatísticas, contra três mercados de uma vez:

1. quantas explicações cada mercado admite sozinho, e quantas sobrevivem aos três;
2. a mesma conta com a **lei compartilhada** e a escala de cada mercado própria, em duas grades;
3. a decomposição: qual estatística é o gargalo da interseção vazia;
4. a família com **duração de episódio heterogênea**, replicada em muitas sementes.

**A regra que o caderno obedece**, e que entrou no contrato depois de me custar um dia: toda
medição sobre mundos sorteados reporta dispersão, ou não reporta nada.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E08_observar_ou_intervir.json.

In [1]:
# <- brinque com: SERIES, GRADE_GROSSA, GRADE_FINA, CONFIGURACOES, SEMENTES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, regimes, volatilidade

RAIZ = Path.cwd()
SERIES = ("sp500.csv", "ibov.csv", "btc.csv")
CHAVES = ("taxa", "pior", "mediana", "acima_do_dobro")
TOLERANCIA = {"taxa": 0.002, "pior": 2.0, "mediana": 1.0, "acima_do_dobro": 0.03}
P, RAZAO, CURTA = 0.05, 3.0, 5.0
GRADE_GROSSA_P = (0.02, 0.05, 0.08, 0.12, 0.20)
GRADE_GROSSA_R = (1.5, 2.0, 2.5, 3.0, 3.5)
GRADE_FINA_P = (0.02, 0.04, 0.06, 0.08, 0.10, 0.14, 0.20, 0.26)
GRADE_FINA_R = (1.5, 1.8, 2.1, 2.4, 2.7, 3.0, 3.5)
PERMANENCIAS = (1.0, 10.0, 30.0, 60.0, 120.0)
CONFIGURACOES = (("unica vinte e cinco", None, None), ("hetero sessenta", 60.0, 0.7),
                 ("hetero cento e vinte", 120.0, 0.7), ("hetero cento e vinte rala", 120.0, 0.3))
SEMENTES = 40
SEMENTE = 211

mercados = {}
for arquivo in SERIES:
    r = volatilidade.retornos_log(dados.carregar_serie(arquivo))
    mercados[arquivo] = {"x": r.to_numpy(), "sigma": float(r.to_numpy().std(ddof=1)),
                         "real": regimes.estatisticas(r.to_numpy())}
    print("%-12s %5d dias | sigma %.5f | taxa %.4f | pior %2d | mediana %.0f | excesso %.4f"
          % (arquivo, len(mercados[arquivo]["x"]), mercados[arquivo]["sigma"],
             mercados[arquivo]["real"]["taxa"], mercados[arquivo]["real"]["pior"],
             mercados[arquivo]["real"]["mediana"], mercados[arquivo]["real"]["acima_do_dobro"]))

sp500.csv     6718 dias | sigma 0.01213 | taxa 0.0513 | pior 20 | mediana 2 | excesso 0.1305
ibov.csv      6620 dias | sigma 0.01691 | taxa 0.0509 | pior 19 | mediana 2 | excesso 0.1038
btc.csv       4387 dias | sigma 0.03486 | taxa 0.0520 | pior 14 | mediana 3 | excesso 0.1138


## O que cada mercado admite sozinho

In [2]:
# O que cada mercado admite sozinho, e o que sobrevive aos tres.
sorteio = np.random.default_rng(SEMENTE)
sobreviventes = {}
for arquivo, m in mercados.items():
    candidatos = []
    for p in GRADE_GROSSA_P:
        for razao in GRADE_GROSSA_R:
            for permanencia in PERMANENCIAS:
                e = regimes.estatisticas(
                    regimes.persistente(len(m["x"]), sorteio, m["sigma"], p, razao, permanencia))
                candidatos.append({"p": p, "razao": razao, "permanencia": permanencia,
                                   "estatisticas": e})
    cabem = regimes.cabem(candidatos, m["real"], TOLERANCIA, CHAVES)
    sobreviventes[arquivo] = {(c["p"], c["razao"], c["permanencia"]) for c in cabem}
    print("%-12s cabem %3d de %d | permanencias %s"
          % (arquivo, len(cabem), len(candidatos), sorted({c["permanencia"] for c in cabem})))
uniao = set().union(*sobreviventes.values())
inter = set.intersection(*sobreviventes.values())
print()
print("em algum mercado: %d explicacoes | em todos: %d" % (len(uniao), len(inter)))

sp500.csv    cabem   0 de 125 | permanencias []


ibov.csv     cabem   3 de 125 | permanencias [10.0, 30.0]


btc.csv      cabem  14 de 125 | permanencias [10.0, 30.0, 60.0, 120.0]

em algum mercado: 16 explicacoes | em todos: 0


## A lei compartilhada, a escala de cada um

In [3]:
# A lei compartilhada (o mecanismo e o mesmo) e a escala de cada mercado propria.
conjunto_que_serve_todos = lambda grade_p, grade_razao: regimes.serve_a_todos(
    mercados, grade_p, grade_razao, PERMANENCIAS, TOLERANCIA, CHAVES, SEMENTE)


grossa = conjunto_que_serve_todos(GRADE_GROSSA_P, GRADE_GROSSA_R)
fina = conjunto_que_serve_todos(GRADE_FINA_P, GRADE_FINA_R)
print("grade grossa: %d de %d triplos servem aos tres" % (len(grossa), 5 * 5 * 5))
print("grade fina:   %d de %d triplos servem aos tres" % (len(fina), 8 * 7 * 5))

grade grossa: 0 de 125 triplos servem aos tres
grade fina:   0 de 280 triplos servem aos tres


## De onde vem o vazio

In [4]:
# De onde vem o vazio: qual estatistica e o gargalo.
passa = {a: [] for a in SERIES}
for p in GRADE_FINA_P:
    for razao in GRADE_FINA_R:
        for permanencia in PERMANENCIAS:
            for arquivo, m in mercados.items():
                s = np.random.default_rng(SEMENTE)
                e = regimes.estatisticas(
                    regimes.persistente(len(m["x"]), s, m["sigma"], p, razao, permanencia))
                passa[arquivo].append(tuple(abs(e[k] - m["real"][k]) <= TOLERANCIA[k] for k in CHAVES))
triplos = len(passa[SERIES[0]])
linhas = []
for i, chave in enumerate(CHAVES):
    sozinho = {a: sum(1 for t in passa[a] if t[i]) for a in SERIES}
    junto = sum(1 for j in range(triplos) if all(passa[a][j][i] for a in SERIES))
    linhas.append({"estatistica": chave, **sozinho, "nos tres": junto})
tabela = pd.DataFrame(linhas).set_index("estatistica")
print("triplos: %d" % triplos)
print(tabela.to_string())
print()
print("as quatro juntas: %s" % {a[:4]: sum(1 for t in passa[a] if all(t)) for a in SERIES})

triplos: 280
                sp500.csv  ibov.csv  btc.csv  nos tres
estatistica                                           
taxa                  194       166      224       145
pior                   34        47       78         2
mediana               280       280      280       280
acima_do_dobro         68       110       78        45

as quatro juntas: {'sp50': 0, 'ibov': 2, 'btc.': 25}


## A família heterogênea, replicada

In [5]:
# A familia de duracao heterogenea, replicada: mediana e dispersao, mercado por mercado.
replicacao = []
for nome, longa, fracao in CONFIGURACOES:
    for arquivo, m in mercados.items():
        piores, excessos, acertos = [], [], 0
        for semente in range(500, 500 + SEMENTES):
            s = np.random.default_rng(semente)
            if longa is None:
                v = regimes.persistente(len(m["x"]), s, m["sigma"], P, RAZAO, 25.0)
            else:
                v = regimes.persistente_heterogeneo(len(m["x"]), s, m["sigma"], P, RAZAO,
                                                    CURTA, longa, fracao)
            e = regimes.estatisticas(v)
            piores.append(e["pior"]); excessos.append(e["acima_do_dobro"])
            if (abs(e["pior"] - m["real"]["pior"]) <= TOLERANCIA["pior"]
                    and abs(e["acima_do_dobro"] - m["real"]["acima_do_dobro"]) <= TOLERANCIA["acima_do_dobro"]):
                acertos += 1
        replicacao.append({"configuracao": nome, "mercado": arquivo,
                           "pior_mediana": float(np.median(piores)),
                           "pior_p10": float(np.percentile(piores, 10)),
                           "pior_p90": float(np.percentile(piores, 90)),
                           "excesso_mediana": float(np.median(excessos)),
                           "excesso_p90": float(np.percentile(excessos, 90)),
                           "acertos_pct": 100.0 * acertos / SEMENTES})
print(pd.DataFrame(replicacao).set_index(["configuracao", "mercado"]).round(3).to_string())

                                     pior_mediana  pior_p10  pior_p90  excesso_mediana  excesso_p90  acertos_pct
configuracao              mercado                                                                               
unica vinte e cinco       sp500.csv          16.0      11.0      18.0            0.080        0.113          7.5
                          ibov.csv           16.0      12.0      18.1            0.082        0.109         45.0
                          btc.csv            14.5      11.9      19.0            0.085        0.111         30.0
hetero sessenta           sp500.csv          17.0      13.9      20.0            0.084        0.098          5.0
                          ibov.csv           17.0      13.8      19.1            0.075        0.101         35.0
                          btc.csv            16.0       9.9      18.1            0.077        0.110         15.0
hetero cento e vinte      sp500.csv          16.0      11.8      19.1            0.066        0.

## As figuras

In [6]:
# Figura 1: o que cada mercado admite, e o que sobrevive aos tres.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
nomes = list(SERIES)
posicoes = np.arange(len(nomes))
eixo.bar(posicoes, [len(sobreviventes[a]) for a in nomes], 0.5, color="#1f4e79")
eixo.axhline(len(inter), color="#b03a2e", ls="--", lw=1.4,
             label="sobrevivem aos tres: %d" % len(inter))
for i, a in enumerate(nomes):
    eixo.annotate("%d" % len(sobreviventes[a]), (i, len(sobreviventes[a])),
                  textcoords="offset points", xytext=(0, 4), ha="center", fontsize=9)
eixo.set_xticks(posicoes)
eixo.set_xticklabels([a.replace(".csv", "") for a in nomes])
eixo.set_ylabel("explicações que cada mercado admite")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E08_observar_ou_intervir", 1)
plt.close(fig)
print("sobreviventes por mercado: %s" % {a[:4]: len(sobreviventes[a]) for a in nomes})

sobreviventes por mercado: {'sp50': 0, 'ibov': 3, 'btc.': 14}


In [7]:
# Figura 2: a replicacao -- mediana do pior bloco e do excesso, contra o dado de cada mercado.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.0))
dados_tabela = pd.DataFrame(replicacao)
for eixo, coluna, alvo, titulo in ((esq, "pior_mediana", "pior", "pior bloco de sessenta dias"),
                                   (dir_, "excesso_mediana", "acima_do_dobro", "excesso de blocos cheios")):
    for i, nome in enumerate([c[0] for c in CONFIGURACOES]):
        valores = [dados_tabela[(dados_tabela["configuracao"] == nome)
                                & (dados_tabela["mercado"] == a)][coluna].iloc[0] for a in SERIES]
        eixo.bar(np.arange(len(SERIES)) + (i - 1.5) * 0.2, valores, 0.2, label=nome if eixo is esq else None)
    eixo.scatter(np.arange(len(SERIES)), [mercados[a]["real"][alvo] for a in SERIES],
                 marker="*", s=140, color="#b03a2e", zorder=5, label="o dado" if eixo is esq else None)
    eixo.set_xticks(np.arange(len(SERIES)))
    eixo.set_xticklabels([a.replace(".csv", "") for a in SERIES], fontsize=9)
    eixo.set_title(titulo, fontsize=10)
    eixo.grid(alpha=0.25, axis="y")
esq.legend(frameon=False, fontsize=7)
fig.tight_layout()
graficos.salvar(fig, "E08_observar_ou_intervir", 2)
plt.close(fig)
print("acertos por configuracao: %s" % dados_tabela.groupby("configuracao")["acertos_pct"].mean().round(1).to_dict())

acertos por configuracao: {'hetero cento e vinte': 13.3, 'hetero cento e vinte rala': 7.5, 'hetero sessenta': 18.3, 'unica vinte e cinco': 27.5}


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Três barras — e a primeira **não existe**: o mercado com mais dados é o que admite
zero explicações, e zero desenhado é indistinguível de dado faltando. As outras duas sobem (três e
catorze), e a linha tracejada da interseção coincide com o eixo horizontal, de modo que a mensagem
central do capítulo é carregada por uma ausência. O que o eixo engana: barra ausente e barra
faltante têm a mesma cara, e a linha do zero colada no eixo some.

**Figura 2.** Dois painéis com as mesmas quatro configurações por mercado, e as estrelas do dado.
As barras ficam abaixo das estrelas em todos os grupos, e quanto mais heterogênea a configuração,
mais abaixo. O que o eixo engana, e vale para todo gráfico de barras com alvo: a barra é uma
mediana e a estrela é um ponto, sem dispersão desenhada — e as faixas de dez a noventa por cento
das configurações se sobrepõem às do dado, de modo que a distância visual entre barra e estrela é
maior do que a distância entre as distribuições.


In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
tabela_gargalo = tabela.reset_index()
dados_replicacao = pd.DataFrame(replicacao)
resultado = {
    "observar_dias_maior": int(max(len(m["x"]) for m in mercados.values())),
    "observar_mercados": len(SERIES),
    "observar_taxa_menor": float(min(m["real"]["taxa"] for m in mercados.values())),
    "observar_taxa_maior": float(max(m["real"]["taxa"] for m in mercados.values())),
    "observar_pior_menor": int(min(m["real"]["pior"] for m in mercados.values())),
    "observar_pior_maior": int(max(m["real"]["pior"] for m in mercados.values())),
    "observar_combinacoes": 5 * 5 * len(PERMANENCIAS),
    "observar_grossa": len(grossa),
    "observar_fina": len(fina),
    "observar_triplos_finos": 8 * 7 * len(PERMANENCIAS),
    "observar_uniao": len(uniao),
    "observar_intersecao": len(inter),
}
for linha in tabela_gargalo.to_dict("records"):
    chave = linha["estatistica"].replace("acima_do_dobro", "excesso")
    resultado["observar_nos_tres_%s" % chave] = int(linha["nos tres"])
for arquivo, curto in zip(SERIES, ("índice", "ibov", "bitcoin")):
    resultado["observar_sobreviventes_%s" % curto] = len(sobreviventes[arquivo])
    resultado["observar_acerto_unica_%s_pct" % curto] = float(
        dados_replicacao[(dados_replicacao["configuracao"] == CONFIGURACOES[0][0])
                         & (dados_replicacao["mercado"] == arquivo)]["acertos_pct"].iloc[0])
    resultado["observar_acerto_hetero_%s_pct" % curto] = float(
        dados_replicacao[(dados_replicacao["configuracao"] == CONFIGURACOES[1][0])
                         & (dados_replicacao["mercado"] == arquivo)]["acertos_pct"].iloc[0])

caminho = Path("lab/resultados/E08_observar_ou_intervir.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E08_observar_ou_intervir.json gravado | 25 grandezas
